# Change Data Feed (CDF) med Delta Sharing

Dette notebooken demonstrerer hvordan du bruker **Change Data Feed (CDF)** til å spore endringer i Delta-tabeller over tid.

In [ ]:
! pip install -r requirements.txt

In [ ]:
import os
import json
import subprocess
import delta_sharing
import delta_sharing
import pandas as pd
from google.cloud import storage

from src.auth import generate_access_token
from src.utils import (
    fetch_config_share,
    create_credentials_config,
)

In [ ]:
project_id = "innsikt-data-dev-ec28"
project_num = "614733074632"
provider_full_identifier = f"projects/{project_num}/locations/global/workloadIdentityPools/skyporten-bi-dev/providers/skyporten-bi-provider-dev"
random_id = "9ivj"
schema_name = "matrikkel_silver_v1_ext"
share_name = f"{random_id}-dev"

CREDENTIALS_PATH = "credentials.json"
TOKEN_PATH = "tmp_maskinporten_token.txt"
CONFIG_PATH = "configs/config.json"
SOURCE_PATH = "share/config.share" #fil

In [ ]:
create_credentials_config(provider_full_identifier, TOKEN_PATH, CREDENTIALS_PATH)

In [ ]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

s = token.get("access_token", "")

print(f"Generated token: {s}")

with open(TOKEN_PATH, 'w') as file:
    file.write(s)

In [ ]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

s = token.get("access_token", "")

print(f"Generated token: {s}")

with open(TOKEN_PATH, 'w') as file:
    file.write(s)

In [ ]:
os.makedirs("share", exist_ok=True)
bucket_id = f"sp-{project_id}-{random_id}"
fetch_config_share(project_id, bucket_id, SOURCE_PATH, CREDENTIALS_PATH)

In [ ]:
# Spesifiser sti til din Delta Sharing config-fil
# Denne filen inneholder credentials og endpoint for din share
CONFIG_FILE = "share/config.share"  # Endre til din config-fil

In [ ]:
# Koble til Delta Sharing
sharing_client = delta_sharing.SharingClient(CONFIG_FILE)
tables = sharing_client.list_all_tables()

if not tables:
    raise Exception("❌ Ingen tabeller funnet i sharen")

# Filtrer bort tabeller som ikke er gode for demonstrasjon
# (kode-tabeller, nøkkel-tabeller, krypterte tabeller)
not_valid_table_names = ["kode", "keys", "encrypted"]
valid_examples_tables = [
    table for table in tables
    if not any(substr in table.name for substr in not_valid_table_names)
]

# Velg første egnede tabell (eller første tabell hvis ingen egnede finnes)
table = valid_examples_tables[0] if len(valid_examples_tables) > 0 else tables[0]

# Bygg full tabell-URL for Delta Sharing
table_url = f"{CONFIG_FILE}#{table.share}.{table.schema}.{table.name}"

print(f"✓ Valgt tabell: {table.name}")
print(f"  Share: {table.share}")
print(f"  Schema: {table.schema}")
print(f"  Full URL: {table_url}")

##  Opprett Spark Session



In [ ]:
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, functions as F

# Stopp eksisterende Spark-session hvis den finnes
try:
    spark.stop()
    print("🔄 Stoppet eksisterende Spark session")
except:
    pass

# Opprett ny Spark-session med Delta Sharing konfigurasjon
spark = (
    SparkSession.builder
    .appName("DeltaSharingCDF")
    .config("spark.jars.packages", "io.delta:delta-sharing-spark_2.12:3.1.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)


# Hent endringer via CDF

1. Beregner et starttidspunkt basert på antall timer tilbake i tid.  
2. Leser endringer fra tabellen siden dette tidspunktet.  
3. Hvis det finnes endringer, grupperes de etter endringstype (`insert`, `update`, `delete`).  
4. Viser en oppsummering av antall endringer per type.  
5. Viser inntil 10 eksempler for hver endringstype.


In [ ]:
lookback_hours = 24
start_ts = (datetime.utcnow() - timedelta(hours=lookback_hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

cdf = (
    spark.read.format("deltaSharing")
    .option("responseFormat", "delta")
    .option("readChangeFeed", "true")
    .option("startingTimestamp", start_ts)
    .load(table_url)
)

if not cdf.rdd.isEmpty():
    change_summary = cdf.groupBy("_change_type").count().orderBy("_change_type")
    change_summary.show(truncate=False)

    for row in change_summary.collect():
        change_type = row["_change_type"]
        cdf.filter(F.col("_change_type") == change_type).show(10, truncate=False)
